# GPT Researcher with Gemini 2.5 - Demo Notebook

This notebook demonstrates how to use GPT Researcher with Google Gemini 2.5 models and Gemini Grounding with Google Search.

## Setup

### Installation Options:

**Option 1: Install in development mode (recommended)**
```bash
# From the repository root directory
pip install -e .
```

**Option 2: Run the cell below to add repo to Python path**
- This allows importing without installation
- Only works while notebook is running

### Requirements:
1. Your Gemini API key from https://aistudio.google.com/app/apikey
2. Required packages: `pip install -r requirements.txt` (from repo root)

## 1. Initial Setup and Configuration

In [1]:
import sys
import os
import re
import asyncio 
from pathlib import Path

# Add repo to path
repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

# Load environment variables
from gpt_researcher import GPTResearcher 
from dotenv import load_dotenv
load_dotenv()

# Verify settings
print(f"✅ RETRIEVER: {os.environ.get('RETRIEVER')}")
print(f"✅ GEMINI_API_KEY: {'Set' if os.environ.get('GEMINI_API_KEY') else 'Missing'}") 


def clean_filename(text):
    # Replace invalid characters with an underscore or empty string
    return re.sub(r'[\\/*?:"<>|]', "", text).strip()

✅ RETRIEVER: gemini_grounding
✅ GEMINI_API_KEY: Set


## 2. Basic Research Example

Let's run a simple research query about current events.

In [2]:
async def basic_research(query, instructions=None):
    """
    Basic research example using Gemini Grounding
    """
    print("🔍 Starting research...\n")
    
    # Create researcher instance
    researcher = GPTResearcher(
        query=query,
        report_type="deep",
        verbose=True
    )
    
    # Conduct research
    print("📚 Gathering information...")
    research_context = await researcher.conduct_research()
    
    # Generate report
    # 3. Switch the report type to "blog_report" before writing
    researcher.report_type = "blog_report"

    # 4. Generate the report (uses deep context + blog prompt + extra instructions if any)
    report = await researcher.write_report(custom_prompt=instructions) 
    
    # Display results
    print("\n" + "="*80)
    print("📄 RESEARCH REPORT")
    print("="*80 + "\n")
    print(report)
    
    # Show sources
    sources = researcher.get_source_urls()
    print("\n" + "="*80)
    print(f"📎 SOURCES ({len(sources)} total)")
    print("="*80)
    for i, source in enumerate(sources, 1):
        print(f"{i}. {source}")
    
    # Show costs
    costs = researcher.get_costs()
    print(f"\n💰 Total cost: ${costs:.4f}")
    
    return report, sources

In [3]:
research_tasks = [
    {
        "query": "Why Field-Level OCR Breaks Down in Real Expense Reimbursement Workflows",
        "instructions": """
Explain how expense documents (receipts, invoices, claims) vary wildly in layout, language, and quality.
Cover issues such as:
- Skewed scans, faded prints, handwritten totals
- Different receipt formats across vendors and countries
- Inconsistent placement of key fields like totals, dates, and tax

Explain why traditional OCR + rules fails to reliably extract fields at scale.
Then explain how DocumentLens solves this by:
- Using layout-aware extraction instead of fixed coordinates
- Grounding each extracted value to its visual location
- Supporting handwritten and printed content together

Conclude with business impact: reduced manual review and cleaner downstream accounting data.
"""
    },
    {
        "query": "Why Converting PDFs to Text Is Not the Same as Understanding a Document",
        "instructions": """
Discuss the misconception that text extraction equals document understanding.
Explain problems caused by:
- Flattened reading order
- Lost section hierarchy
- Tables turned into unreadable text blocks

Describe how DocumentLens approaches document parsing differently by:
- Preserving sections, tables, and semantic boundaries
- Outputting structured Markdown or JSON instead of raw text
- Enabling documents to be programmatically navigated and analyzed

Include examples like contracts, reports, and multi-column documents.
"""
    },
    {
        "query": "The Hidden Risks of Manual Contract Comparison in Legal Teams",
        "instructions": """
Explain how legal teams often rely on manual or line-based text diff tools.
Highlight risks:
- Missing semantic changes
- Inability to compare scanned vs digital documents
- High review time and human error

Show how DocumentLens Document Comparison:
- Aligns documents structurally and semantically
- Identifies insertions, deletions, and modifications clearly
- Works across formats and scan qualities

Emphasize time savings and reduced legal risk.
"""
    },
    {
        "query": "Why Handwritten Content Is Still a Major Blind Spot for OCR Systems",
        "instructions": """
Explain why handwriting remains difficult:
- Variability in writing styles
- Mixed handwriting and printed text
- Low-quality scans

Discuss how most OCR systems treat handwriting as an edge case.
Explain how DocumentLens addresses this by:
- Treating handwriting as a first-class signal
- Combining visual understanding with contextual reasoning
- Extracting meaning, not just characters

Use examples like receipts, forms, and annotations.
"""
    },
    {
        "query": "Why Southeast Asian Documents Confuse Global OCR Platforms",
        "instructions": """
Explain regional challenges:
- Mixed languages in a single document
- Local numbering, naming, and formatting conventions
- Country-specific document types

Describe why globally trained OCR systems struggle.
Then explain how DocumentLens is built specifically for Southeast Asia:
- Pre-trained on local document types
- Language-aware extraction
- Cultural context understanding

Highlight accuracy and reduced customization effort.
"""
    },
    {
        "query": "From Scanned Contracts to Structured Clauses: Closing the Gap",
        "instructions": """
Explain the difficulty of extracting clauses from scanned or image-based contracts.
Discuss problems with:
- Lost structure
- Clauses split across pages
- Headings detached from content

Explain how DocumentLens:
- Preserves document hierarchy
- Extracts clauses as structured units
- Keeps traceability to original locations

Focus on legal and compliance use cases.
"""
    },
    {
        "query": "Why Table Extraction Is One of the Hardest Problems in Document AI",
        "instructions": """
Explain common table extraction failures:
- Merged rows and columns
- Broken headers
- Loss of relationships between cells

Describe how DocumentLens:
- Understands table structure visually
- Preserves row-column relationships
- Outputs tables as structured data

Use examples from finance, reporting, and analytics.
"""
    },
    {
        "query": "The Cost of Downstream Data Cleaning After OCR",
        "instructions": """
Explain how OCR errors propagate downstream:
- Broken ERP imports
- Manual reconciliation
- BI dashboards built on incorrect data

Show how DocumentLens reduces downstream correction by:
- Producing schema-aligned outputs
- Grounding data to source locations
- Improving field-level accuracy

Quantify operational impact where possible.
"""
    },
    {
        "query": "Why Multi-Language Documents Require More Than Language Detection",
        "instructions": """
Explain why detecting language alone is insufficient.
Discuss issues with:
- Mixed-language paragraphs
- Language-specific layouts
- Semantic ambiguity

Explain how DocumentLens:
- Processes content at a segment level
- Understands context across languages
- Extracts unified structured outputs

Use examples from invoices, forms, and government documents.
"""
    },
    {
        "query": "Why Visual Elements Matter in Financial and Legal Documents",
        "instructions": """
Discuss non-text elements:
- Stamps, seals, signatures
- Charts and figures
- Checkboxes and markings

Explain how traditional OCR ignores these.
Explain how DocumentLens:
- Detects and interprets visual elements
- Converts charts and figures into structured data
- Preserves evidence for audit and verification

Focus on compliance and risk management.
"""
    },
    {
        "query": "Why Reading Order Determines Data Accuracy",
        "instructions": """
Explain how incorrect reading order leads to:
- Misaligned fields
- Incorrect associations
- Broken summaries

Describe how DocumentLens:
- Determines logical reading flow
- Maintains contextual grouping
- Prevents cross-section contamination

Use examples from multi-column reports.
"""
    },
    {
        "query": "The Problem with Rule-Based Extraction at Enterprise Scale",
        "instructions": """
Explain why rules and templates fail:
- High maintenance cost
- Poor adaptability
- Fragility to layout changes

Explain how DocumentLens replaces rules with:
- Layout-aware intelligence
- Schema-driven extraction
- Model-based generalization

Focus on scalability and long-term cost reduction.
"""
    },
    {
        "query": "Why OCR Accuracy Metrics Alone Are Misleading",
        "instructions": """
Explain why character accuracy ≠ business accuracy.
Discuss:
- Correct text in wrong fields
- High OCR scores with unusable outputs

Explain how DocumentLens focuses on:
- Field accuracy
- Structural correctness
- Downstream usability

Tie results to real workflows.
"""
    },
    {
        "query": "From Images to Insights: Why Image Understanding Matters",
        "instructions": """
Explain how images carry meaning beyond text.
Discuss charts, diagrams, and figures.
Explain how DocumentLens:
- Interprets visual data
- Converts it into structured insights
- Enables analytics and automation

Use BI and reporting examples.
"""
    },
    {
        "query": "Why Government and Regulatory Documents Require Traceability",
        "instructions": """
Explain compliance requirements:
- Audit trails
- Source verification
- Accountability

Explain how DocumentLens:
- Grounds every field to its source
- Enables visual verification
- Supports compliance workflows

Use public sector examples.
"""
    },
    {
        "query": "Why High-Volume Document Processing Fails Without Structure",
        "instructions": """
Explain scaling issues when structure is lost.
Discuss operational bottlenecks.
Explain how DocumentLens:
- Produces machine-ready outputs
- Enables automation at scale
- Reduces human intervention

Focus on throughput and reliability.
"""
    },
    {
        "query": "Why Document Parsing Is Foundational to AI Agents",
        "instructions": """
Explain why AI agents need structured inputs.
Discuss limitations of raw text.
Explain how DocumentLens:
- Produces agent-ready structured data
- Enables reasoning over documents
- Supports downstream AI workflows

Position parsing as infrastructure.
"""
    },
    {
        "query": "Why Image Forgery Detection Is Becoming a Core Document Capability",
        "instructions": """
Explain growing risks of document manipulation.
Discuss use cases in compliance and fraud.
Explain how DocumentLens image forgery detection:
- Analyzes image integrity
- Supports verification workflows
- Complements extraction and parsing

Position it as trust infrastructure.
"""
    },
    {
        "query": "Why One-Size-Fits-All OCR Fails in Enterprise Environments",
        "instructions": """
Explain diversity of enterprise documents.
Discuss why generic tools underperform.
Explain how DocumentLens:
- Adapts across document types
- Supports customization without rules
- Integrates into enterprise systems

Conclude with flexibility and ROI.
"""
    },
    {
        "query": "From Documents to Systems: Closing the Automation Loop",
        "instructions": """
Explain the gap between documents and systems like ERP and CRM.
Explain how DocumentLens:
- Outputs structured, validated data
- Integrates via APIs
- Enables end-to-end automation

Frame DocumentLens as a bridge, not just an OCR tool.
"""
    }
]

In [4]:
output_folder = "content"
os.makedirs(output_folder, exist_ok=True)

print(f"Starting research on {len(research_tasks)} topics...")

# 4. Loop through the tasks
for i, task in enumerate(research_tasks, 1):
	query = task["query"]
	instructions = task["instructions"]
	
	print(f"[{i}/{len(research_tasks)}] Researching: {query}...")

	try:
		# Run the actual research
		# Note: We await here so it finishes one before starting the next
		report, sources = await basic_research(query, instructions=instructions)
		
		# Create a safe filename based on the query
		safe_name = clean_filename(query)
		file_path = os.path.join(output_folder, f"{safe_name}.md")

		# Save the report
		with open(file_path, "w", encoding="utf-8") as f:
			f.write(report)
		
		print(f"   Saved to: {file_path}")

	except Exception as e:
		print(f"   ERROR processing '{query}': {e}")

print("\nAll tasks completed.")

Starting research on 20 topics...
[1/20] Researching: Why Field-Level OCR Breaks Down in Real Expense Reimbursement Workflows...
🔍 Starting research...

📚 Gathering information...

🔍 DEEP RESEARCH: Starting with breadth=2, depth=1, concurrency=4
Searching with Gemini Grounding: Why Field-Level OCR Breaks Down in Real Expense Reimbursement Workflows
Resolving 10 Vertex AI redirect URLs to original sources...
Found 10 grounded results from Gemini.

📊 DEEP RESEARCH: depth=1, breadth=2, query=
        Initial Query: Why Field-Level OCR Breaks Down in Real Expense Reimbursement Workflows
Foll...
🔎 Generating 2 search queries...
✅ Generated 2 queries: ['"benchmarking AI OCR field-level extraction accuracy on complex expense documents including international line-items and dynamic digital receipts"', '"Total Cost of Ownership (TCO) analysis of human-in-the-loop (HITL) in automated expense management workflows considering user trust and exception handling costs"']


INFO:     [18:35:13] 🔍 Starting the research task for '"Total Cost of Ownership (TCO) analysis of human-in-the-loop (HITL) in automated expense management workflows considering user trust and exception handling costs"'...
INFO:     [18:35:13] 📈 Business Analyst Agent
INFO:     [18:35:13] 🌐 Browsing the web to learn more about the task: "Total Cost of Ownership (TCO) analysis of human-in-the-loop (HITL) in automated expense management workflows considering user trust and exception handling costs"...


Searching with Gemini Grounding: "Total Cost of Ownership (TCO) analysis of human-in-the-loop (HITL) in automated expense management workflows considering user trust and exception handling costs"
Resolving 10 Vertex AI redirect URLs to original sources...


INFO:     [18:35:24] 🤔 Planning the research strategy and subtasks...
INFO:     [18:35:24] 🔍 Starting the research task for '"benchmarking AI OCR field-level extraction accuracy on complex expense documents including international line-items and dynamic digital receipts"'...
INFO:     [18:35:24] 🤖 AI Research Agent
INFO:     [18:35:24] 🌐 Browsing the web to learn more about the task: "benchmarking AI OCR field-level extraction accuracy on complex expense documents including international line-items and dynamic digital receipts"...


Found 10 grounded results from Gemini.
Searching with Gemini Grounding: "benchmarking AI OCR field-level extraction accuracy on complex expense documents including international line-items and dynamic digital receipts"
Resolving 10 Vertex AI redirect URLs to original sources...


...

Found 5 grounded results from Gemini.


INFO:     [21:40:44] 📄 Scraped 5 pages of content
INFO:     [21:40:44] 🖼️ Selected 4 new images from 19 total images
INFO:     [21:40:44] 🌐 Scraping complete
INFO:     [21:40:44] 📚 Getting relevant content based on query: technological milestones in AI continuous learning and data extraction for closed-loop automation systems...
INFO:     [21:40:46] 📚 Combined research context: 0 MCP sources, web content
INFO:     [21:40:46] Finalized research step.
💸 Total Research Costs: $0.01149868
INFO:     [21:40:58] ✍️ Writing report for 'From Documents to Systems: Closing the Automation Loop'...


# From Documents to Systems: Closing the Automation Loop

In today's rapidly
 evolving technological landscape, the journey **from documents to systems: closing the automation loop** represents a fundamental shift in how organizations operate. We are moving beyond simple task automation to creating intelligent, adaptive systems that continuously learn, act, and refine their processes. This paradigm shift, driven by advancements in Artificial Intelligence (AI) and robotics, promises unprecedented levels of efficiency, robustness, and autonomy. The core of this transformation lies in establishing seamless, closed-loop feedback mechanisms that allow AI systems to not only process information but also to understand context, make decisions, execute actions, and learn from the outcomes, often with critical human oversight.

## The Evolution of Automation: Beyond Open-Loop Systems

Historically, many automated systems, particularly those integrating
 Large Language Models (LLMs), have operate

INFO:     [21:41:42] 📝 Report written for 'From Documents to Systems: Closing the Automation Loop'



📄 RESEARCH REPORT

# From Documents to Systems: Closing the Automation Loop

In today's rapidly evolving technological landscape, the journey **from documents to systems: closing the automation loop** represents a fundamental shift in how organizations operate. We are moving beyond simple task automation to creating intelligent, adaptive systems that continuously learn, act, and refine their processes. This paradigm shift, driven by advancements in Artificial Intelligence (AI) and robotics, promises unprecedented levels of efficiency, robustness, and autonomy. The core of this transformation lies in establishing seamless, closed-loop feedback mechanisms that allow AI systems to not only process information but also to understand context, make decisions, execute actions, and learn from the outcomes, often with critical human oversight.

## The Evolution of Automation: Beyond Open-Loop Systems

Historically, many automated systems, particularly those integrating Large Language Models (L

## 3. Multi-Retriever Research

Combine Gemini Grounding with other search engines for comprehensive results.

In [ ]:
async def multi_retriever_research():
    """
    Research using multiple retrievers for comprehensive results
    """
    # Configure multiple retrievers
    os.environ["RETRIEVER"] = "gemini_grounding,duckduckgo"
    
    print("🔍 Starting multi-retriever research...")
    print(f"Using retrievers: {os.environ['RETRIEVER']}\n")
    
    researcher = GPTResearcher(
        query="What are the latest breakthroughs in quantum computing?",
        report_type="research_report",
        verbose=True
    )
    
    context = await researcher.conduct_research()
    report = await researcher.write_report()
    
    print("\n" + "="*80)
    print("📄 MULTI-RETRIEVER RESEARCH REPORT")
    print("="*80 + "\n")
    print(report)
    
    sources = researcher.get_source_urls()
    print(f"\n📎 Found {len(sources)} sources across multiple retrievers")
    
    return report

# Run multi-retriever research
multi_report = await multi_retriever_research()

## 4. Fast Mode Research

Optimize for speed by disabling thinking and using faster models.

In [ ]:
async def fast_research():
    """
    Fast research with thinking disabled
    """
    import time
    
    # Configure for speed
    os.environ["RETRIEVER"] = "gemini_grounding"
    os.environ["GEMINI_THINKING_BUDGET"] = "0"  # Disable thinking
    
    print("⚡ Starting FAST research (thinking disabled)...\n")
    
    start_time = time.time()
    
    researcher = GPTResearcher(
        query="Quick summary of latest tech news this week",
        report_type="research_report",
        verbose=True
    )
    
    context = await researcher.conduct_research()
    report = await researcher.write_report()
    
    elapsed_time = time.time() - start_time
    
    print("\n" + "="*80)
    print("📄 FAST RESEARCH REPORT")
    print("="*80 + "\n")
    print(report)
    
    print(f"\n⏱️ Completed in: {elapsed_time:.2f} seconds")
    print(f"💰 Cost: ${researcher.get_costs():.4f}")
    
    return report, elapsed_time

# Run fast research
fast_report, duration = await fast_research()

## 5. Quality Mode Research

Enable thinking for higher quality analysis (slower but more thorough).

In [ ]:
async def quality_research():
    """
    High-quality research with thinking enabled
    """
    import time
    
    # Configure for quality
    os.environ["RETRIEVER"] = "gemini_grounding"
    os.environ.pop("GEMINI_THINKING_BUDGET", None)  # Use default (thinking enabled)
    
    print("🎯 Starting QUALITY research (thinking enabled)...\n")
    
    start_time = time.time()
    
    researcher = GPTResearcher(
        query="What are the implications of recent AI safety research?",
        report_type="research_report",
        verbose=True
    )
    
    context = await researcher.conduct_research()
    report = await researcher.write_report()
    
    elapsed_time = time.time() - start_time
    
    print("\n" + "="*80)
    print("📄 QUALITY RESEARCH REPORT")
    print("="*80 + "\n")
    print(report)
    
    print(f"\n⏱️ Completed in: {elapsed_time:.2f} seconds")
    print(f"💰 Cost: ${researcher.get_costs():.4f}")
    
    return report, elapsed_time

# Run quality research
quality_report, quality_duration = await quality_research()

## 6. Detailed Research with Source Analysis

Examine the research sources and metadata in detail.

In [ ]:
async def detailed_research_with_sources():
    """
    Research with detailed source analysis
    """
    os.environ["RETRIEVER"] = "gemini_grounding"
    os.environ["GEMINI_THINKING_BUDGET"] = "0"
    
    print("🔬 Starting detailed research with source analysis...\n")
    
    researcher = GPTResearcher(
        query="What are the latest developments in climate technology?",
        report_type="research_report",
        verbose=True
    )
    
    # Conduct research
    context = await researcher.conduct_research()
    report = await researcher.write_report()
    
    # Get detailed source information
    sources = researcher.get_research_sources()
    
    print("\n" + "="*80)
    print("📄 RESEARCH REPORT")
    print("="*80 + "\n")
    print(report)
    
    # Detailed source analysis
    print("\n" + "="*80)
    print(f"📚 DETAILED SOURCE ANALYSIS ({len(sources)} sources)")
    print("="*80 + "\n")
    
    for i, source in enumerate(sources, 1):
        print(f"\n[Source {i}]")
        print(f"Title: {source.get('title', 'N/A')}")
        print(f"URL: {source.get('url', 'N/A')}")
        
        # Show snippet of content
        content = source.get('raw_content', '')
        if content:
            snippet = content[:200] + "..." if len(content) > 200 else content
            print(f"Content Preview: {snippet}")
        
        # Show images if available
        images = source.get('image_urls', [])
        if images:
            print(f"Images: {len(images)} found")
        
        print("-" * 80)
    
    # Get research context
    research_context = researcher.get_research_context()
    print(f"\n📊 Research Context Items: {len(research_context)}")
    
    return report, sources

# Run detailed research
detail_report, detail_sources = await detailed_research_with_sources()

## 7. Custom Query with Specific Configuration

Create a fully customized research task.

In [ ]:
async def custom_research():
    """
    Fully customized research configuration
    """
    # Reset to single retriever
    os.environ["RETRIEVER"] = "gemini_grounding"
    os.environ["GEMINI_THINKING_BUDGET"] = "0"
    
    print("⚙️ Starting custom configured research...\n")
    
    # Custom configuration
    researcher = GPTResearcher(
        query="YOUR CUSTOM QUERY HERE",
        report_type="research_report",  # Options: research_report, outline_report, etc.
        report_format="markdown",       # Options: markdown, apa, etc.
        tone="Objective",                # Tone of the report
        max_subtopics=5,                 # Max subtopics to explore
        verbose=True                     # Show detailed logs
    )
    
    # Conduct research
    context = await researcher.conduct_research()
    
    # Generate report
    report = await researcher.write_report()
    
    print("\n" + "="*80)
    print("📄 CUSTOM RESEARCH REPORT")
    print("="*80 + "\n")
    print(report)
    
    # Additional information
    print(f"\n📊 Statistics:")
    print(f"  - Sources: {len(researcher.get_source_urls())}")
    print(f"  - Context items: {len(researcher.get_research_context())}")
    print(f"  - Total cost: ${researcher.get_costs():.4f}")
    
    return report

# Uncomment to run custom research
# custom_report = await custom_research()

## 8. Comparison: Gemini Grounding vs Other Retrievers

Compare results from different search engines.

In [ ]:
async def compare_retrievers():
    """
    Compare research results from different retrievers
    """
    query = "What are the key features of Gemini 2.0?"
    
    results = {}
    
    # Test Gemini Grounding
    print("🔍 Testing Gemini Grounding...\n")
    os.environ["RETRIEVER"] = "gemini_grounding"
    os.environ["GEMINI_THINKING_BUDGET"] = "0"
    
    researcher1 = GPTResearcher(query=query, report_type="research_report", verbose=False)
    await researcher1.conduct_research()
    report1 = await researcher1.write_report()
    
    results["Gemini Grounding"] = {
        "report": report1[:500] + "...",
        "sources": len(researcher1.get_source_urls()),
        "cost": researcher1.get_costs()
    }
    
    # Test DuckDuckGo
    print("\n🦆 Testing DuckDuckGo...\n")
    os.environ["RETRIEVER"] = "duckduckgo"
    
    researcher2 = GPTResearcher(query=query, report_type="research_report", verbose=False)
    await researcher2.conduct_research()
    report2 = await researcher2.write_report()
    
    results["DuckDuckGo"] = {
        "report": report2[:500] + "...",
        "sources": len(researcher2.get_source_urls()),
        "cost": researcher2.get_costs()
    }
    
    # Display comparison
    print("\n" + "="*80)
    print("📊 RETRIEVER COMPARISON")
    print("="*80 + "\n")
    
    for retriever, data in results.items():
        print(f"\n{retriever}:")
        print(f"  Sources: {data['sources']}")
        print(f"  Cost: ${data['cost']:.4f}")
        print(f"  Report Preview: {data['report'][:150]}...")
        print("-" * 80)
    
    return results

# Run comparison
comparison_results = await compare_retrievers()

## 9. Export Reports

Save your research reports to files.

In [ ]:
async def export_research():
    """
    Conduct research and export to file
    """
    os.environ["RETRIEVER"] = "gemini_grounding"
    os.environ["GEMINI_THINKING_BUDGET"] = "0"
    
    print("📝 Conducting research for export...\n")
    
    researcher = GPTResearcher(
        query="Summary of recent breakthroughs in renewable energy",
        report_type="research_report",
        verbose=True
    )
    
    context = await researcher.conduct_research()
    report = await researcher.write_report()
    
    # Export to markdown file
    output_file = "research_report.md"
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("# Research Report\n\n")
        f.write(report)
        f.write("\n\n## Sources\n\n")
        for i, source in enumerate(researcher.get_source_urls(), 1):
            f.write(f"{i}. {source}\n")
    
    print(f"\n✅ Report exported to: {output_file}")
    print(f"📊 Report length: {len(report)} characters")
    print(f"💰 Total cost: ${researcher.get_costs():.4f}")
    
    return output_file

# Export research
output_file = await export_research()
print(f"\n📄 You can now read the report in: {output_file}")

## 10. Tips and Best Practices

### Configuration Tips

1. **For Speed**: Set `GEMINI_THINKING_BUDGET=0`
2. **For Quality**: Leave thinking budget unset (default)
3. **For Current Events**: Use `gemini_grounding` retriever
4. **For Comprehensive Research**: Use multiple retrievers
5. **For JavaScript Sites**: Use `SCRAPER=browser`

### Cost Optimization

- Disable thinking for production systems
- Use flash-lite model for simple queries
- Limit max_subtopics for focused research
- Monitor costs with `researcher.get_costs()`

### Troubleshooting

If you encounter issues:
1. Check your API key is set correctly
2. Verify `google-genai` package is installed
3. Enable verbose mode: `verbose=True`
4. Check the logs for specific error messages

### Additional Resources

- Documentation: https://docs.gptr.dev
- Gemini Setup Guide: See `GEMINI_SETUP.md` in repo root
- Gemini API Docs: https://ai.google.dev/gemini-api/docs

## Summary

This notebook demonstrated:
- ✅ Basic research with Gemini Grounding
- ✅ Multi-retriever research
- ✅ Fast vs Quality mode
- ✅ Detailed source analysis
- ✅ Custom configurations
- ✅ Retriever comparisons
- ✅ Report exporting

You're now ready to use GPT Researcher with Gemini 2.5! 🚀